In [6]:
from pathlib import Path
from time import perf_counter
import importlib

import Numbers_Diary_Parser_v1 as diary_parser
diary_parser = importlib.reload(diary_parser)

# Detailed operations stay in the .py file. The notebook keeps the visible workflow.
parse_numbers_file = diary_parser.parse_numbers_file
dry_run_numbers_file = diary_parser.dry_run_numbers_file
list_month_folders = diary_parser.list_month_folders
list_numbers_files = diary_parser.list_numbers_files
configure_run_logging = diary_parser.configure_run_logging
finish_run_logging = diary_parser.finish_run_logging


In [7]:
# RUN MODE
# "dry_run": open/read every Numbers file and exercise parser reads, but write no Diary Export data.
# "parse":    perform the real transactional CSV + IMG + HTML export.
RUN_MODE = "parse"

# Hand-pick the year folders to process.
SOURCE_YEAR_FOLDERS = [
    Path(
        "/Users/huohsien/Library/Mobile Documents/"
        "com~apple~Numbers/Documents/2026 Diary"
    ),
]

OUTPUT_ROOT = Path("/Users/huohsien/Downloads/Diary Export")
CONTINUE_ON_ERROR = True

# Optional test filter. Leave as None for the whole selected year(s).
# Example: ONLY_FILE_NAMES = {"July 27.numbers", "July 29.numbers"}
ONLY_FILE_NAMES = None

# ONLY_FILE_NAMES = {
#     "October 11.numbers",
# }

# Run logs belong to this Git project, not to derived Diary Export data.
PROJECT_ROOT = Path(diary_parser.__file__).resolve().parent
LOG_DIR = PROJECT_ROOT / "logs"
RUN_LOGGER, RUN_LOG_PATH = configure_run_logging(
    run_mode=RUN_MODE,
    log_dir=LOG_DIR,
)

print("RUN_MODE:", RUN_MODE)
print("Run log:", RUN_LOG_PATH)


RUN_MODE: parse
Run log: /Users/huohsien/workspace/python/Diary conversion from Numbers to Text and Image Files/logs/diary-parser-20261016-122557-parse.log


In [8]:
# CHECKPOINT 1 — show exactly what the notebook is going to traverse.
# No parsing and no Diary Export writing happens in this cell.
for year_folder in SOURCE_YEAR_FOLDERS:
    print(f"\nYEAR: {year_folder}")
    RUN_LOGGER.info("DISCOVERY year=%s", year_folder)

    month_folders = list_month_folders(year_folder)
    for month_folder in month_folders:
        numbers_files = list_numbers_files(month_folder)
        if ONLY_FILE_NAMES is not None:
            numbers_files = [
                path for path in numbers_files
                if path.name in ONLY_FILE_NAMES
            ]
        print(f"  {month_folder.name}: {len(numbers_files)} .numbers files")
        RUN_LOGGER.info(
            "DISCOVERY month=%s files=%d",
            month_folder,
            len(numbers_files),
        )



YEAR: /Users/huohsien/Library/Mobile Documents/com~apple~Numbers/Documents/2026 Diary
  January: 31 .numbers files
  February: 28 .numbers files
  March: 31 .numbers files
  April: 30 .numbers files
  May: 32 .numbers files
  June: 30 .numbers files
  July: 31 .numbers files
  August: 31 .numbers files
  September: 30 .numbers files
  October: 16 .numbers files


In [9]:
# RUN — year/month/file loops stay here intentionally so the workflow remains visible.
results = []
errors = []
batch_timer_started = perf_counter()

RUN_LOGGER.info("BATCH begin mode=%s", RUN_MODE)

for year_folder in SOURCE_YEAR_FOLDERS:
    print(f"\n===== {year_folder.name} =====")

    month_folders = list_month_folders(year_folder)

    for month_folder in month_folders:
        numbers_files = list_numbers_files(month_folder)
        if ONLY_FILE_NAMES is not None:
            numbers_files = [
                path for path in numbers_files
                if path.name in ONLY_FILE_NAMES
            ]
        print(f"\n--- {month_folder.name}: {len(numbers_files)} files ---")

        for file_index, numbers_file in enumerate(numbers_files, start=1):
            print(f"[{file_index}/{len(numbers_files)}] {numbers_file.name}")

            try:
                if RUN_MODE == "dry_run":
                    result = dry_run_numbers_file(numbers_file)
                elif RUN_MODE == "parse":
                    result = parse_numbers_file(numbers_file, OUTPUT_ROOT)
                else:
                    raise ValueError(
                        f"Unknown RUN_MODE={RUN_MODE!r}; use 'dry_run' or 'parse'."
                    )

                result["status"] = "OK"
                results.append(result)

            except Exception as exc:
                error = {
                    "year_folder": str(year_folder),
                    "month_folder": str(month_folder),
                    "source": str(numbers_file),
                    "status": "ERROR",
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                }
                errors.append(error)
                print(f"  ERROR: {type(exc).__name__}: {exc}")
                RUN_LOGGER.error(
                    "BATCH file failed source=%s error_type=%s error=%s",
                    numbers_file,
                    type(exc).__name__,
                    exc,
                )

                if not CONTINUE_ON_ERROR:
                    raise

batch_elapsed_seconds = finish_run_logging(
    run_mode=RUN_MODE,
    started_at=batch_timer_started,
    results=results,
    errors=errors,
)



===== 2026 Diary =====

--- January: 31 files ---
[1/31] January 1.numbers
[2/31] January 2.numbers
[3/31] January 3.numbers
[4/31] January 4.numbers
[5/31] January 5.numbers
[6/31] January 6.numbers
[7/31] January 7.numbers
[8/31] January 8.numbers
[9/31] January 9.numbers
[10/31] January 10.numbers
[11/31] January 11.numbers
[12/31] January 12.numbers
[13/31] January 13.numbers
[14/31] January 14.numbers
[15/31] January 15.numbers
[16/31] January 16.numbers
[17/31] January 17.numbers
[18/31] January 18.numbers
[19/31] January 19.numbers
[20/31] January 20.numbers
[21/31] January 21.numbers
[22/31] January 22.numbers
[23/31] January 23.numbers
[24/31] January 24.numbers
[25/31] January 25.numbers
[26/31] January 26.numbers
[27/31] January 27.numbers
[28/31] January 28.numbers
[29/31] January 29.numbers
[30/31] January 30.numbers
[31/31] January 31.numbers

--- February: 28 files ---
[1/28] February 1.numbers
[2/28] February 2.numbers
[3/28] February 3.numbers
[4/28] February 4.number

In [10]:
# CHECKPOINT 2 — batch summary.
hours, remainder = divmod(batch_elapsed_seconds, 3600)
minutes, seconds = divmod(remainder, 60)
elapsed_text = f"{int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}"

print(f"mode={RUN_MODE}")
print(f"parsed_ok={len(results)}  failed={len(errors)}")
print(f"elapsed={elapsed_text}")
print(f"log={RUN_LOG_PATH}")

if errors:
    print("\nErrors:")
    for error in errors:
        print(
            f"- {error['source']}\n"
            f"  {error['error_type']}: {error['error']}"
        )
else:
    print("No parsing errors.")


mode=parse
parsed_ok=290  failed=0
elapsed=00:01:18.15
log=/Users/huohsien/workspace/python/Diary conversion from Numbers to Text and Image Files/logs/diary-parser-20261016-122557-parse.log
No parsing errors.
